# CNN-Based Semiconductor Wafer Defect Detection — Colab Quickstart

**AI 570 — Team 4** (Paul, Pyle, Rajan, Rettura)

End-to-end training on the WM-811K dataset with a free Colab T4 GPU. Wall-clock: ~20 minutes for all three models at 20 epochs.

## Before you start

1. **Enable GPU.** Menu bar → `Runtime` → `Change runtime type` → `Hardware accelerator: T4 GPU` → `Save`.
2. **Get the dataset.** You need `LSWMD_new.pkl` (~2.2 GB). Pick one option (do this BEFORE running cell 4 below):
   - **Option A (recommended):** upload `LSWMD_new.pkl` to your Google Drive at `MyDrive/datasets/LSWMD_new.pkl`
   - **Option B:** use the Colab upload dialog inside the notebook (slow — up to ~15 min over home broadband)
   - **Option C:** download from Kaggle inside Colab (needs `kaggle.json` API token — instructions below)

Then just run each cell top-to-bottom. The dataset cell auto-detects whichever option you chose.

## Cell 1. Clone repo and install

In [ ]:
REPO_URL = 'https://github.com/bpyle02/CNN-Based-Semiconductor-Wafer-Defect-Detection-Using-the-WM-811K-Dataset.git'
REPO_DIR = 'CNN-Based-Semiconductor-Wafer-Defect-Detection-Using-the-WM-811K-Dataset'

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull --ff-only
!pip install -q -e '.[dev]'
print('\nInstall complete. Restart runtime NOW if prompted, then re-run from here.')

## Cell 2. Verify GPU

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Total VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    !nvidia-smi | head -15
else:
    raise RuntimeError('No GPU detected. Menu: Runtime -> Change runtime type -> T4 GPU, then re-run this cell.')

## Cell 3. Get the dataset into `data/LSWMD_new.pkl`

This cell tries each option in order and stops as soon as one succeeds:

1. Already present at `data/LSWMD_new.pkl` (skip)
2. Google Drive `MyDrive/datasets/LSWMD_new.pkl` — copies it in
3. Colab upload dialog — prompts you to pick the file from your computer
4. Kaggle — uploads your `kaggle.json` and pulls from the public dataset

**Most students will use option 2 (Drive).** Put `LSWMD_new.pkl` anywhere in your Drive and edit the `DRIVE_DATASET_PATH` below to point at it.

In [ ]:
import os, shutil

DATASET_PATH = 'data/LSWMD_new.pkl'
DRIVE_DATASET_PATH = '/content/drive/MyDrive/datasets/LSWMD_new.pkl'

os.makedirs('data', exist_ok=True)

def dataset_ready(path=DATASET_PATH, min_bytes=1_000_000_000):
    return os.path.exists(path) and os.path.getsize(path) > min_bytes

# Option 1: already present
if dataset_ready():
    print(f'Dataset already at {DATASET_PATH} ({os.path.getsize(DATASET_PATH)/1e9:.2f} GB)')

# Option 2: Google Drive
elif True:
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        if os.path.exists(DRIVE_DATASET_PATH):
            print(f'Copying from {DRIVE_DATASET_PATH} ...')
            shutil.copy(DRIVE_DATASET_PATH, DATASET_PATH)
            print(f'Done. Dataset size: {os.path.getsize(DATASET_PATH)/1e9:.2f} GB')
        else:
            print(f'NOT found at {DRIVE_DATASET_PATH}. Falling through to upload option.')
    except Exception as e:
        print(f'Drive mount/copy failed: {e}')

# Option 3: browser upload (only if still not ready)
if not dataset_ready():
    print('Opening upload dialog — select LSWMD_new.pkl from your computer (slow, ~10–15 min)...')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        name = list(uploaded.keys())[0]
        shutil.move(name, DATASET_PATH)
        print(f'Uploaded {name} -> {DATASET_PATH} ({os.path.getsize(DATASET_PATH)/1e9:.2f} GB)')

assert dataset_ready(), (
    'Dataset is not present. Options:\n'
    '  a) Put LSWMD_new.pkl at MyDrive/datasets/ in Google Drive and re-run this cell.\n'
    '  b) Use the upload dialog above.\n'
    '  c) See Cell 3b below for Kaggle download.'
)

### Cell 3b (optional). Kaggle download

Skip this if Cell 3 already loaded the dataset. To use this option:

1. Go to <https://www.kaggle.com/settings>, click **Create New Token** — this downloads `kaggle.json`.
2. Run this cell and upload `kaggle.json` when prompted.
3. The WM-811K public dataset will be downloaded and unpacked.

Note: the Kaggle download is the **raw** WM-811K (`LSWMD.pkl`, 1.9 GB), not the team's preprocessed `LSWMD_new.pkl`. The loader in `src/data/dataset.py` handles both.

In [ ]:
# import os
# from google.colab import files
# uploaded = files.upload()   # upload kaggle.json
# os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
# os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
# os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
# !pip install -q kaggle
# !kaggle datasets download -d qingyi/wm811k-wafer-map -p data/ --unzip
# # The file unpacks as data/LSWMD.pkl; rename if needed:
# import shutil
# if os.path.exists('data/LSWMD.pkl') and not os.path.exists('data/LSWMD_new.pkl'):
#     shutil.copy('data/LSWMD.pkl', 'data/LSWMD_new.pkl')
# print('size:', round(os.path.getsize('data/LSWMD_new.pkl')/1e9, 2), 'GB')

## Cell 4. Sanity-check the dataset

In [ ]:
from src.data import load_dataset
df = load_dataset('data/LSWMD_new.pkl')
print('shape:', df.shape)
print('columns:', df.columns.tolist())
# Use failureClass (extracted string labels), NOT failureType (raw nested
# arrays that hang value_counts because they aren't hashable).
if 'failureClass' in df.columns:
    print('\nFailure class distribution:')
    print(df['failureClass'].value_counts())

## Cell 5. Run the test suite (~1 min on T4)

In [ ]:
!pytest -q --maxfail=5

## Cell 6. Train all three models on GPU

Expected wall-clock on T4 at batch size 64, 20 epochs: ~25–35 minutes total.

On Colab free tier (12.7 GB RAM) this cell trains each model in a separate subprocess so RAM is released between models. If you have Colab Pro with more RAM you can bump `BATCH_SIZE` to 128 or switch to a single `--model all` call.

If you hit `CUDA out of memory`, drop `BATCH_SIZE` to 32.

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64   # T4-safe; Colab free tier has 12.7 GB RAM. Bump to 128 only if you've confirmed RAM headroom.
import os, gc, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Free memory left over from the earlier cells (df, test imports, etc.)
# so the training subprocess starts with a clean slate.
for _name in ['df', 'm', 'models', 'ensemble', 'gradcam']:
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

# Train one model at a time in separate subprocesses. This is critical on
# Colab free tier: `--model all` keeps the full dataset + every model in
# one long-lived Python process, which can exceed the 12.7 GB RAM cap on
# the transition from ResNet-18 to EfficientNet-B0. Running each as its
# own process releases RAM back to the OS between models.
for _model in ['cnn', 'resnet', 'efficientnet']:
    print(f'\n===== training {_model} =====\n')
    !python train.py --model {_model} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --device cuda

## Cell 7. Inspect results

In [ ]:
import os, json, glob

metric_files = sorted(glob.glob('results/*metrics.json'))
if not metric_files:
    print('No metrics.json files found in results/. Did training complete?')
for path in metric_files:
    with open(path) as f:
        m = json.load(f)
    acc = m.get('accuracy', m.get('test_accuracy', 0))
    macro = m.get('macro_f1', m.get('test_macro_f1', 0))
    weighted = m.get('weighted_f1', m.get('test_weighted_f1', 0))
    print(f'{os.path.basename(path):40s}  acc={acc:.4f}  macro_f1={macro:.4f}  weighted_f1={weighted:.4f}')

In [ ]:
from IPython.display import Image, display
import glob
imgs = sorted(glob.glob('results/*_confusion_matrix.png') + glob.glob('results/*_training_curves.png'))
if not imgs:
    print('No result images found.')
for path in imgs:
    print(path)
    display(Image(path))

## Cell 8. Save checkpoints and results to Drive

Colab free sessions disconnect after idle/disconnect events. Run this before closing the tab.

In [ ]:
import os, shutil, time

if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

out_dir = f"/content/drive/MyDrive/wafer_runs/{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out_dir, exist_ok=True)

for src_dir in ['checkpoints', 'results']:
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, os.path.join(out_dir, src_dir), dirs_exist_ok=True)
print('Saved checkpoints and results to:', out_dir)

---

**Done.** Inspect the confusion matrices and per-class F1 scores — macro-F1 is the imbalance-aware metric to report; accuracy is dominated by the 85% `none` class.